In [17]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

# Create an API client

client = Anthropic()
model = "claude-haiku-4-5"

In [7]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(
        **params
    )
    
    return response.content[0].text

In [8]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

chat(messages) # el problema es que lo generado tiene backtips, json, enters y en general caracteres que no interesan


'```json\n{\n  "Name": "MyEventRule",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n```'

In [9]:
messages = []

# Para solucionar, agregaremos una secuencia de parada. Básicamente le estamos diciendo a claude
# que en el mensaje de asistente ya está el inicio del json
# y le estamos diciendo que para el mensaje cuando encuentre la secuencia de los backtips
# así que solo nos quedamos con el contenido del json.
# esta estrategia no está disponible para modelos como sonnet 4.6 u opus


add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [10]:
messages

[{'role': 'user',
  'content': 'Generate a very short event bridge rule as json'},
 {'role': 'assistant', 'content': '```json'}]

In [11]:
import json

json.loads(text.strip())

{'Name': 'MySimpleRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

# Exercise

In [12]:
messages = []

prompt = """
Benerate three different sample AWS CLI commands. Each should be very short
"""

add_user_message(messages, prompt)

text = chat(messages)
text

'# Three AWS CLI Commands\n\n1. **List all S3 buckets**\n```bash\naws s3 ls\n```\n\n2. **Describe EC2 instances**\n```bash\naws ec2 describe-instances\n```\n\n3. **Get current AWS account ID**\n```bash\naws sts get-caller-identity\n```'

In [14]:
from IPython.display import Markdown

Markdown(text)

# Three AWS CLI Commands

1. **List all S3 buckets**
```bash
aws s3 ls
```

2. **Describe EC2 instances**
```bash
aws ec2 describe-instances
```

3. **Get current AWS account ID**
```bash
aws sts get-caller-identity
```

In [29]:
messages = []

prompt = """
Benerate three different sample AWS CLI commands. Each should be very short
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

text = chat(messages, stop_sequences=["```"])
text

'\naws ec2 describe-instances --region us-east-1\naws s3 ls s3://my-bucket/\naws iam list-users\n'

In [30]:
Markdown(text)


aws ec2 describe-instances --region us-east-1
aws s3 ls s3://my-bucket/
aws iam list-users
